In [1]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime
#from google.colab import userdata

c:\Users\Rohit kumar\Desktop\Rohit_ningthoujam\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#Chunking

In [2]:
from pathlib import Path
pdf_folder_location = Path("tesla-annual-reports").resolve()
print("Loading PDFs from:", pdf_folder_location)

Loading PDFs from: C:\Users\Rohit kumar\Desktop\Rohit_ningthoujam\tesla-annual-reports


In [3]:
pdf_loader = PyPDFDirectoryLoader(str(pdf_folder_location))

In [4]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [6]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [7]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [8]:
len(tesla_10k_chunks)

3337

In [9]:
print(tesla_10k_chunks)

[Document(metadata={'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2022-05-02T10:10:26+00:00', 'title': '', 'source': 'C:\\Users\\Rohit kumar\\Desktop\\Rohit_ningthoujam\\tesla-annual-reports\\tsla-10ka_20211231-gen.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1'}, page_content='UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K/A\n(Amendment\tNo.\t1)\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\tfiscal\tyear\tended\t\nDecember\t31,\t\n2021\n\t\nOR\n☐\nTRANSITION\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\ttransition\tperiod\tfrom\t\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\n\tto\t\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\nCommission\tFile\tNumber:\t\n001-34756\n\t\nTesla,\tInc.\n(Exact\tname\tof\tregistrant\tas\tspecified\tin\tits\tcharter)\n\t\n\t\nDelaware\n\

# Database Creation


In [10]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [12]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Rohit kumar\AppData\Local\Temp\ipykernel_16540\4093584008.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [13]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [14]:
chromadb_client.heartbeat()

1780552384064831500

In [15]:
chromadb_client.count_collections()

1

In [16]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [17]:
chromadb_client.count_collections()

1

In [20]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023']

In [18]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [19]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    batch = tesla_10k_chunks[i:i+500] # Get the current batch of up to 500 chunks
    ids = ["text_" + str(j) for j in range(i, i + len(batch))] # Assign unique IDs to each chunk in the batch
    vectorstore.add_documents( # Add documents to the vector store in batches
        documents=batch,
        ids=ids
    )

    i += len(batch) # Increment the index by the size of the current batch
    time.sleep(0.5) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [20]:
collection = chromadb_client.get_collection(tesla_10k_collection)

In [21]:
# Count the number of records in the collection
collection.count()

3337

In [22]:
# Inspect the first 10 records
collection.peek()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['text_0',
  'text_1',
  'text_2',
  'text_3',
  'text_4',
  'text_5',
  'text_6',
  'text_7',
  'text_8',
  'text_9'],
 'embeddings': array([[ 0.02675908,  0.00757637, -0.04422267, ..., -0.03966741,
         -0.09228931, -0.0373573 ],
        [ 0.01642387, -0.06928593, -0.02833171, ..., -0.01646161,
         -0.06955732, -0.01540411],
        [ 0.00760543, -0.01997787, -0.00596519, ..., -0.0130257 ,
         -0.06449894, -0.02837417],
        ...,
        [ 0.03221607,  0.0295096 , -0.04174366, ..., -0.02659888,
         -0.04579795,  0.0019708 ],
        [ 0.05646931,  0.12058288, -0.0257278 , ..., -0.05004434,
         -0.03446646, -0.00241624],
        [ 0.0119407 ,  0.02874993, -0.01575188, ...,  0.01867358,
         -0.0381767 ,  0.00357348]]),
 'documents': ['UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K/A\n(Amendment\tNo.\t1)\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES

In [23]:
# tables / keys present in the vector DB
collection.peek().keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'data', 'metadatas', 'included'])

In [24]:
# Inspect a specific record

collection.get(
    ids=['text_999']
)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['text_999'],
 'embeddings': None,
 'documents': ['term,\tassuming\tall\tother\trevenue\trecognition\tcriteria\thave\tbeen\tmet.\tThe\tdifference\tbetween\tthe\tpayments\treceived\tand\tthe\trevenue\trecognized\tis\trecorded\tas\ndeferred\trevenue\tor\tdeferred\tasset\ton\tthe\tconsolidated\tbalance\tsheet.\nFor\tsolar\tenergy\tsystems\twhere\tcustomers\tpurchase\telectricity\tfrom\tus\tunder\tPPAs\tprior\tto\tJanuary\t1,\t2019,\twe\thave\tdetermined\tthat\tthese\tagreements\nshould\tbe\taccounted\tfor\tas\toperating\tleases\tpursuant\tto\tASC\t840.\tRevenue\tis\trecognized\tbased\ton\tthe\tamount\tof\telectricity\tdelivered\tat\trates\tspecified\tunder\nthe\tcontracts,\tassuming\tall\tother\trevenue\trecognition\tcriteria\tare\tmet.\nWe\trecord\tas\tdeferred\trevenue\tany\tamounts\tthat\tare\tcollected\tfrom\tcustomers,\tincluding\tlease\tprepayments,\tin\texcess\tof\trevenue\trecognized\tand\noperations\tand\tmaintenance\tservice\tfees,\twhich\tis\trecognized\tas\trevenue\tra

## Retrieving relevant records

In [25]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [26]:
user_query = "Automotive revenue in 2021?"

In [27]:
retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='text_3106', metadata={'creationdate': '2024-01-29T11:11:14+00:00', 'creator': 'wkhtmltopdf 0.12.6', 'page': 38, 'page_label': '39', 'producer': 'Qt 5.15.2', 'source': 'C:\\Users\\Rohit kumar\\Desktop\\Rohit_ningthoujam\\tesla-annual-reports\\tsla-20231231-gen.pdf', 'title': '', 'total_pages': 130}, page_content='2022\n2021\n$\n%\n$\n%\nAutomotive\tsales\n$\n78,509\t\n$\n67,210\t\n$\n44,125\t\n$\n11,299\t\n17\t\n%\n$\n23,085\t\n52\t\n%\nAutomotive\tregulatory\tcredits\n1,790\t\n1,776\t\n1,465\t\n14\t\n1\t\n%\n311\t\n21\t\n%\nAutomotive\tleasing\n2,120\t\n2,476\t\n1,642\t\n(356)\n(14)\n%\n834\t\n51\t\n%\nTotal\tautomotive\trevenues\n82,419\t\n71,462\t\n47,232\t\n10,957\t\n15\t\n%\n24,230\t\n51\t\n%\nServices\tand\tother\n8,319\t\n6,091\t\n3,802\t\n2,228\t\n37\t\n%\n2,289\t\n60\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\nrevenue\n90,738\t\n77,553\t\n51,034\t\n13,185\t\n17\t\n%\n26,519\t\n52\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\trevenue\n6,035\t\n3,9

In [28]:
len(retriever.invoke(user_query))

5

#RAG Q and A

In [29]:
import chromadb

from langchain_chroma import Chroma

#from google.colab import userdata


In [30]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from groq import Groq
client = Groq()

In [31]:
model_name = 'openai/gpt-oss-120b'

In [32]:
### Before running the command ensure the numpy version is 2.3.4. If not then upgrade numpy as per previous steps in notebook.
### Then do not restart the notebook. If you restart then you will loose all loaded variables. Just click Cancel instad of Restart

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [33]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [34]:
chromadb_client

In [35]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [36]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [37]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [38]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002B6AE7E6BD0>, search_kwargs={'k': 5})

#Prompt design

In [155]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [156]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [157]:
user_query = "What was the automotive revenue in 2021?"

In [158]:
relevant_document_chunks = retriever.invoke(user_query)

In [159]:
len(relevant_document_chunks)

3

In [160]:
print(vectorstore_persisted._collection.count())

3337


In [161]:
for document in relevant_document_chunks:
    print(document)
    break


page_content='2022
2021
$
%
$
%
Automotive	sales
$
78,509	
$
67,210	
$
44,125	
$
11,299	
17	
%
$
23,085	
52	
%
Automotive	regulatory	credits
1,790	
1,776	
1,465	
14	
1	
%
311	
21	
%
Automotive	leasing
2,120	
2,476	
1,642	
(356)
(14)
%
834	
51	
%
Total	automotive	revenues
82,419	
71,462	
47,232	
10,957	
15	
%
24,230	
51	
%
Services	and	other
8,319	
6,091	
3,802	
2,228	
37	
%
2,289	
60	
%
Total	automotive	&	services	and	other	segment
revenue
90,738	
77,553	
51,034	
13,185	
17	
%
26,519	
52	
%
Energy	generation	and	storage	segment	revenue
6,035	
3,909	
2,789	
2,126	
54	
%
1,120	
40	
%
Total	revenues
$
96,773	
$
81,462	
$
53,823	
$
15,311	
19	
%
$
27,639	
51	
%
Automotive	&	Services	and	Other	Segment
Automotive	sales	revenue	includes	revenues	related	to	cash	and	financing	deliveries	of	new	Model	S,	Model	X,	Semi,	Model	3,	Model	Y,	and
Cybertruck	vehicles,	including	access	to	our	FSD	Capability	features	and	their	ongoing	maintenance,	internet	connectivity,	free	Supercharging	programs
and	ov

In [176]:
i = 0
for document in relevant_document_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

-------Chunk 0-------
2022
2021
$
%
$
%
Automotive sales
$
78,509 
$
67,210 
$
44,125 
$
11,299 
17 
%
$
23,085 
52 
%
Automotive regulatory credits
1,790 
1,776 
1,465 
14 
1 
%
311 
21 
%
Automotive leasing
2,120 
2,476 
1,642 
(356)
(14)
%
834 
51 
%
Total automotive revenues
82,419 
71,462 
47,232 
10,957 
15 
%
24,230 
51 
%
Services and other
8,319 
6,091 
3,802 
2,228 
37 
%
2,289 
60 
%
Total automotive & services and other segment
revenue
90,738 
77,553 
51,034 
13,185 
17 
%
26,519 
52 
%
Energy generation and storage segment revenue
6,035 
3,909 
2,789 
2,126 
54 
%
1,120 
40 
%
Total revenues
$
96,773 
$
81,462 
$
53,823 
$
15,311 
19 
%
$
27,639 
51 
%
Automotive & Services and Other Segment
Automotive sales revenue includes revenues related to cash and financing deliveries of new Model S, Model X, Semi, Model 3, Model Y, and
Cybertruck vehicles, including access to our FSD Capability features and their ongoing maintenance, internet connectivity, free Supercharging program

# Composing response

In [182]:
user_query = "What was the automotive revenue in 2021?"

In [183]:
user_query = "What fines or penalties were imposed on the company in 2022?"

In [184]:
user_query = "What are the risk factors mentioned in the 10k report?"


In [185]:
user_query = "What regulatory investigations and legal actions against the company are mentioned in the report?"

In [187]:
model_name = 'openai/gpt-oss-120b'

docs = retriever.invoke(user_query)
#docs = vectorstore_persisted.similarity_search_with_score(user_query, k=5)
context_list = [doc.page_content for doc in docs]
#context_list = [d.page_content for d in docs]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

The report notes that the company is cooperating with ongoing government investigations (referenced in Note 15, Commitments and Contingencies) and acknowledges the possibility of future legal actions by the SEC, the U.S. Department of Justice, or other governmental agencies. It also states that, to date, no agency has concluded that any wrongdoing occurred.


In [191]:
baseline_top_chunks = []

for idx, doc in enumerate(docs):

    baseline_top_chunks.append({
        "chunk_id": f"chunk_{idx+1}",
        "section": "Unknown",
        "year": 2023
    })

# RAG Q and  A  ANd Improving with QUERY EXPANSION

In [193]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [194]:
model_name = 'openai/gpt-oss-120b'

In [196]:
prompt=[
    {'role':'system', 'content':query_expansion_system_message},
    {'role':'user',   'content': user_message_template.format(
        question=user_query
    )}
]

In [197]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nWhat regulatory investigations and legal actions against the company are mentioned in the report?\n</Question>\n'}]

In [198]:
query_expansions = client.chat.completions.create(model=model_name,
                                                  messages=prompt,
                                                  temperature=0)

In [199]:
print(query_expansions.choices[0].message.content)

What regulatory investigations and legal actions against the company are mentioned in the report?
Which regulatory probes and lawsuits involving the company are disclosed in the filing?
Can you list any government investigations and legal proceedings against the company noted in the report?


In [200]:
query_expansions = client.chat.completions.create(model=model_name,
                                                  messages=prompt,
                                                  temperature=0)

In [201]:
print(query_expansions)

ChatCompletion(id='chatcmpl-c127b9cc-6e79-4b37-9a72-a9c3854cee2f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='What regulatory investigations and legal actions against the company are mentioned in the report?\nWhich regulatory probes and lawsuits involving the company are disclosed in the filing?\nCan you list any government investigations and legal proceedings against the company noted in the report?', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user wants query expansion: produce at least 3 versions of the question, each on a new line, no numbering or bullets, no extra text. Provide different phrasings/synonyms. The original: "What regulatory investigations and legal actions against the company are mentioned in the report?" Need variations: "Which regulatory investigations and lawsuits are referenced in the filing?" "What legal proceedings and regulatory probes involving the compa

In [202]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [203]:
print(query_expansions.choices[-1].message.reasoning)

The user wants query expansion: produce at least 3 versions of the question, each on a new line, no numbering or bullets, no extra text. Provide different phrasings/synonyms. The original: "What regulatory investigations and legal actions against the company are mentioned in the report?" Need variations: "Which regulatory investigations and lawsuits are referenced in the filing?" "What legal proceedings and regulatory probes involving the company are disclosed in the 10-K?" "Can you list any government investigations and legal actions against the company noted in the report?" Provide at least 3. Ensure each line is a question. No numbering.


In [204]:
len(query_expansions_list)

3

In [205]:
query_expansions_list

['What regulatory investigations and legal actions against the company are mentioned in the report?',
 'Which regulatory probes and lawsuits involving the company are disclosed in the filing?',
 'Can you list any government investigations and legal proceedings against the company noted in the report?']

In [206]:
expanded_context_list = []

In [207]:
expanded_context_list = []
expanded_top_chunks = []

for query in query_expansions_list:

    # docs = vectorstore_persisted.similarity_search_with_score(
    #     query,
    #     k=3
    # )
        docs = retriever.invoke(query)
        for doc  in docs:
            expanded_context_list.append(
            doc.page_content
        )

        expanded_top_chunks.append({
            "chunk_id": f"page_{doc.metadata['page']}",
            "section": "Unknown",
            "year": 2023,
            "retrieved_by": query
        })

In [208]:
citations = []

seen = set()

for chunk in expanded_top_chunks:

    if chunk["chunk_id"] not in seen:

        seen.add(chunk["chunk_id"])

        citations.append({
            "chunk_id": chunk["chunk_id"],
            "source_doc": "tsla-20231231-gen.pdf",
            "section": chunk["section"],
            "year": chunk["year"]
        })

In [209]:
len(expanded_context_list)

9

In [210]:
expanded_context_list

['Table\tof\tContents\nauthority\tinvestigations,\tlegal\tproceedings\tby\tinternational\tgovernmental\tentities\tor\tothers\tresulting\tin\tmandated\tdisclosure\tof\tsensitive\tdata\tor\tother\ncommercially\tunfavorable\tterms.\tNotwithstanding\tour\tefforts\tto\tprotect\tthe\tsecurity\tand\tintegrity\tof\tour\tcustomers’\tpersonal\tinformation,\twe\tmay\tbe\nrequired\tto\texpend\tsignificant\tresources\tto\tcomply\twith\tdata\tbreach\trequirements\tif,\tfor\texample,\tthird\tparties\timproperly\tobtain\tand\tuse\tthe\tpersonal\ninformation\tof\tour\tcustomers\tor\twe\totherwise\texperience\ta\tdata\tloss\twith\trespect\tto\tthe\tpersonal\tinformation\twe\tprocess\tand\thandle.\tA\tmajor\tbreach\tof\nour\tnetwork\tsecurity\tand\tsystems\tmay\toccur\tdespite\tdefensive\tmeasures,\tand\tmay\tresult\tin\tfines,\tpenalties\tand\tdamages\tand\tharm\tour\tbrand,\tprospects\nand\toperating\tresults.\nWe\tcould\tbe\tsubject\tto\tliability,\tpenalties\tand\tother\trestrictive\tsanctions\tand\t

In [211]:
final_context_documents = set(expanded_context_list)

In [212]:
len(final_context_documents) 

8

In [213]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [214]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [215]:
model_name = 'openai/gpt-oss-120b'

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=final_context_documents,
         question=user_query
        )
    }
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)


The report references several regulatory and legal matters involving the company:

- **Government investigations** – The company says it is cooperating with certain government investigations (as noted in Note 15, “Commitments and Contingencies”). It adds that, to its knowledge, no agency has concluded that any wrongdoing occurred, but it could face liability, penalties or other sanctions if the SEC, the U.S. Department of Justice or any other government agency were to bring legal action in the future.

- **Court judgment** – The report notes that on October 16 2018 the U.S. District Court for the Southern District of New York entered a final judgment approving the terms of a settlement (the specific matter is not detailed in the excerpt).

- **Statement of no pending actions** – The company also states that, to its knowledge, there are no current legal actions, lawsuits, arbitrations, administrative or other proceedings, charges, complaints, investigations, inspections, audits, or noti

In [216]:
import json
import os

json_file = "query_expansion_results.json"

if os.path.exists(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        data = [data]
else:
    data = []

question_id = f"Q{len(data) + 1}"

output_schema = {
    "question_id": question_id,
    "original_query": user_query,
    "expanded_queries": query_expansions_list,
    "baseline_top_chunks": baseline_top_chunks,
    "expanded_top_chunks": expanded_top_chunks,
    "final_answer": prediction,
    "citations": citations,
    "retrieval_improvement_analysis":
        "Query expansion improved retrieval coverage compared to baseline retrieval."
}

data.append(output_schema)

with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)